In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Select your NDD and the date
ndd = 'VAS'
date = 'JUNE_25_2026'

In [ ]:
#! pwd

In [ ]:
# Select the NDD case files created in step 01
cases = pd.read_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/NDD_rule_of_two/{ndd}_with_tenure_{date}.csv')
cases

In [ ]:
#Load controls created in step 02
controls = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/CONTROLS_with_tenure_JUNE_25_2026.csv')
controls = controls.drop(columns = 'DEM_DATE')
controls

In [ ]:
# Combine cases and controls
df = pd.concat([cases, controls])

#Check to make sure no duplicate IDs
print(df.ID.value_counts())

df = df.sort_values(by = f'{ndd}_DATE')
df = df.drop_duplicates(subset = 'ID', keep = 'first')

#Check to make sure no duplicate IDs
print(df.ID.value_counts())

df

In [ ]:
#Check number of cases and controls
df[f'{ndd}_DATE'].isna().value_counts()

# Adding bacterial codes

In [ ]:
condition_list = ['K11_ORAL', 'J10_INFLUPNEU', 'AB1_OTHER_BACTERIAL', 'K11_APPENDIX', 'AB1_BACTINF_NOS', 'J10_PNEUMOBACT', 'AB1_OTHER_SPIROCHAETAL', 'AB1_BACT_INTEST_OTH', 'O15_PUERP_SEPSIS', 'M13_REACTARTH', 'AB1_BACT_BIR_OTHER_INF_AGENTS', 'AB1_TUBERCULOSIS', 'M13_PYOGARTH', 'AB1_SALMONELLA_OTH', 'G6_MENINGBACT', 'AB1_SYPHILIS', 'AB1_SEQULAE_TUBERCU', 'AB1_ZOONOTIC_BACTERIAL', 'P16_BACTERIAL_SEPSIS_NEWBO', 'AB1_TYPHOIDPARATHYPHOID', 'AB1_ CHOLERA']
print(condition_list)
print(len(condition_list))

In [ ]:
# test one code
condition = 'J10_INFLUPNEU'
test = pd.read_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/ICD10_Codes/Finngen_codes/{condition}.csv')
test = test[['ID', condition]]
test

In [ ]:
for condition in condition_list:
    test = pd.read_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/ICD10_Codes/Finngen_codes/{condition}.csv')
    test = test[['ID', condition]]
    df = df.merge(test, left_on = 'ID', right_on = 'ID', how = 'left')

In [ ]:
df[f'{ndd}_DATE'].isna().value_counts()

In [ ]:
#Encode NDD to 1 or 0
df[ndd] = np.where(df[ndd + '_DATE'].isna(), 0, 1)

#GENETIC_SEX to 1 or 2
df.loc[df.sex_at_birth == 'Female', 'SEX'] = '0'
df.loc[df.sex_at_birth == 'Male', 'SEX'] = '1'

In [ ]:
# Because we only have reliable data since 2015, the longer the study could be is 9 years
START_DATE = '2015-01-01'

for code in condition_list:
    df['cutoff_date1'] = pd.to_datetime(df['tenure_date']) - pd.DateOffset(years=1)
    df['cutoff_date5'] = pd.to_datetime(df['tenure_date']) - pd.DateOffset(years=5)

    #Select drug data at ANY time before tenure
    df[f'QC0_{code}'] = np.where((pd.to_datetime(df[f'{code}']) < pd.to_datetime(df['tenure_date'])), 1, 0)

    # # Select drug data at 0–1 years before tenure
    df[f'QC0_1_{code}'] = np.where((pd.to_datetime(df[f'{code}']) < pd.to_datetime(df['tenure_date'])) & (pd.to_datetime(df[f'{code}']) >= pd.to_datetime(df['cutoff_date1'])), 1, 0)

    # Select drug data at 1–5 years before tenure
    df[f'QC1_5_{code}'] = np.where(
        (pd.to_datetime(df[f'{code}']) >= pd.to_datetime(df['cutoff_date5'])) & (pd.to_datetime(df[f'{code}']) < pd.to_datetime(df['cutoff_date1'])), 1, 0)

        # #Select data only 5+ years before study end
    df[f'QC5_{code}'] = np.where((df[f'{code}'] < df['cutoff_date5']), 1, 0)

In [ ]:
df

In [ ]:
test = df[['ID', 'tenure_date', 'K11_ORAL', 'QC0_K11_ORAL', 'QC0_1_K11_ORAL', 'QC1_5_K11_ORAL', 'QC5_K11_ORAL']]
test = test[~test['K11_ORAL'].isna()]
#test.QC0_1_K11_ORAL.value_counts()
#test = test[test['QC0_1_K11_ORAL'] == 1]
test

In [ ]:
df.sex_at_birth.value_counts()

# Add genetic status - APOE

In [ ]:
apoe = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/other/APOE_genotypes')
#eliminate unknown samples
apoe = apoe[apoe['APOE_GENOTYPE'] != 'unknown']
apoe

In [ ]:
apoe.APOE_GENOTYPE.value_counts()

In [ ]:
apoe["APOE"] = apoe["APOE_GENOTYPE"].map({
    "e3/e4": 1,
    "e4/e4": 2
}).fillna(0).astype(int)

In [ ]:
apoe = apoe[['IID', 'APOE']]
apoe = apoe.rename(columns = {'IID':'ID'})
apoe

In [ ]:
df = df.merge(apoe, left_on = 'ID', right_on = 'ID', how = 'left')
df

In [ ]:
df.APOE.value_counts(dropna=False)

In [ ]:
df = df[~df['APOE'].isna()]
df

In [ ]:
df.VAS_DATE.value_counts()

In [ ]:
date = 'JULY_1_2026'
df.to_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/coxfiles/{ndd}_{date}_ready_cox.csv', header = True, index = False)